# 🏛️ ADOIT Data Product Pipeline
## Domain: Enterprise Architecture | Catalog: `adoit_product` | Owner: EA Team

Complete **Bronze → Silver → Gold** pipeline for the Application Landscape data product.

**Pipeline Flow:**
```
ADOIT CSVs → Bronze (raw_applications, raw_capabilities, raw_map)
                ↓
             Silver (applications, business_capabilities) — enriched
                ↓
             Gold (application_landscape) — joined 360° view
```

## 🪨 Bronze Layer — Raw Ingestion from ADOIT
Raw data as extracted from ADOIT REST API. No transformations.

In [ ]:
%sql
-- Load raw applications into bronze
INSERT OVERWRITE adoit_product.bronze.raw_applications VALUES
  ('APP-001', 'SAP ERP Central', 'Dr. Mueller', 'Finance', 'Active', 'Critical', 'SAP S/4HANA Cloud', '2018-03-15'),
  ('APP-002', 'ServiceNow ITSM', 'K. Schmidt', 'IT Operations', 'Active', 'Critical', 'ServiceNow SaaS', '2020-01-10'),
  ('APP-003', 'Salesforce CRM', 'M. Weber', 'Sales', 'Active', 'High', 'Salesforce Cloud', '2019-06-01'),
  ('APP-004', 'Legacy Mainframe HR', 'P. Fischer', 'Human Resources', 'Retiring', 'Medium', 'COBOL/DB2 Mainframe', '1998-11-20'),
  ('APP-005', 'Azure Data Platform', 'S. Rajashekar', 'Data & Analytics', 'Active', 'Critical', 'Databricks/Azure/Delta Lake', '2023-01-15'),
  ('APP-006', 'Microsoft 365', 'A. Wagner', 'Corporate', 'Active', 'High', 'Microsoft 365 Cloud', '2021-04-01'),
  ('APP-007', 'Custom CMS', 'L. Becker', 'Marketing', 'Active', 'Low', 'PHP/MySQL On-Prem', '2015-08-10'),
  ('APP-008', 'Jira Project Mgmt', 'T. Hoffmann', 'Engineering', 'Active', 'Medium', 'Atlassian Cloud', '2019-09-15'),
  ('APP-009', 'Oracle EBS Finance', 'H. Schulz', 'Finance', 'Retiring', 'High', 'Oracle EBS On-Prem', '2005-02-28'),
  ('APP-010', 'Power BI Analytics', 'R. Koch', 'Data & Analytics', 'Active', 'Medium', 'Power BI/Azure Cloud', '2022-06-15')

In [ ]:
%sql
-- Load raw business capabilities into bronze
INSERT OVERWRITE adoit_product.bronze.raw_business_capabilities VALUES
  ('CAP-001', 'Financial Planning & Analysis', 'Finance', 'Critical', 'Advanced'),
  ('CAP-002', 'IT Service Management', 'IT Operations', 'High', 'Established'),
  ('CAP-003', 'Customer Relationship Mgmt', 'Sales', 'High', 'Advanced'),
  ('CAP-004', 'Human Capital Management', 'Human Resources', 'Medium', 'Emerging'),
  ('CAP-005', 'Data Analytics & BI', 'Data & Analytics', 'Critical', 'Advanced'),
  ('CAP-006', 'Enterprise Collaboration', 'Corporate', 'Medium', 'Established'),
  ('CAP-007', 'Content Management', 'Marketing', 'Low', 'Emerging'),
  ('CAP-008', 'Project Portfolio Mgmt', 'Engineering', 'Medium', 'Established'),
  ('CAP-009', 'Financial Accounting', 'Finance', 'Critical', 'Advanced'),
  ('CAP-010', 'Workforce Analytics', 'Human Resources', 'Medium', 'Emerging')

In [ ]:
%sql
-- Load raw app-capability mapping into bronze
INSERT OVERWRITE adoit_product.bronze.raw_app_capability_map VALUES
  ('APP-001', 'CAP-001', 'Primary'),
  ('APP-001', 'CAP-009', 'Primary'),
  ('APP-002', 'CAP-002', 'Primary'),
  ('APP-003', 'CAP-003', 'Primary'),
  ('APP-004', 'CAP-004', 'Primary'),
  ('APP-005', 'CAP-005', 'Primary'),
  ('APP-005', 'CAP-010', 'Secondary'),
  ('APP-006', 'CAP-006', 'Primary'),
  ('APP-007', 'CAP-007', 'Primary'),
  ('APP-008', 'CAP-008', 'Primary'),
  ('APP-009', 'CAP-001', 'Secondary'),
  ('APP-009', 'CAP-009', 'Secondary'),
  ('APP-010', 'CAP-005', 'Secondary')

In [ ]:
%sql
-- Verify bronze: Applications
SELECT * FROM adoit_product.bronze.raw_applications ORDER BY app_id

In [ ]:
%sql
-- Verify bronze: Business capabilities
SELECT * FROM adoit_product.bronze.raw_business_capabilities ORDER BY capability_id

## ⚙️ Silver Layer — Curated & Quality-Scored
Transformations: standardization, enrichment (age, cloud flag), quality scoring.

In [ ]:
%sql
MERGE INTO adoit_product.silver.applications AS target
USING (
    SELECT 
        app_id, TRIM(app_name) AS app_name, app_owner,
        INITCAP(department) AS department, lifecycle_status, criticality, technology_stack,
        go_live_date,
        ROUND(DATEDIFF(CURRENT_DATE(), go_live_date) / 365.25, 1) AS app_age_years,
        CASE 
            WHEN LOWER(technology_stack) LIKE '%cloud%' OR LOWER(technology_stack) LIKE '%saas%'
                 OR LOWER(technology_stack) LIKE '%azure%' OR LOWER(technology_stack) LIKE '%databricks%' THEN true
            ELSE false
        END AS is_cloud_hosted,
        ROUND((CASE WHEN app_owner IS NOT NULL THEN 0.25 ELSE 0 END +
               CASE WHEN department IS NOT NULL THEN 0.25 ELSE 0 END +
               CASE WHEN lifecycle_status IS NOT NULL THEN 0.25 ELSE 0 END +
               CASE WHEN criticality IS NOT NULL THEN 0.25 ELSE 0 END), 2) AS quality_score,
        current_timestamp() AS processed_at
    FROM adoit_product.bronze.raw_applications
    WHERE app_id IS NOT NULL
) AS source
ON target.app_id = source.app_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [ ]:
%sql
MERGE INTO adoit_product.silver.business_capabilities AS target
USING (
    SELECT capability_id, capability_name, capability_domain,
           business_value, maturity_level, current_timestamp() AS ingested_at
    FROM adoit_product.bronze.raw_business_capabilities
) AS source
ON target.capability_id = source.capability_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [ ]:
%sql
SELECT app_id, app_name, app_owner, department, lifecycle_status, criticality,
       is_cloud_hosted, app_age_years, quality_score
FROM adoit_product.silver.applications ORDER BY quality_score DESC

## 🏆 Gold Layer — Application Landscape Data Product
Joins applications with business capabilities for a 360° portfolio view.

In [ ]:
%sql
MERGE INTO adoit_product.gold.application_landscape AS target
USING (
    SELECT 
        a.app_id, a.app_name, a.app_owner, a.department,
        a.lifecycle_status, a.criticality, a.technology_stack,
        a.is_cloud_hosted, a.app_age_years,
        COUNT(DISTINCT m.capability_id) AS capability_count,
        CONCAT_WS(', ', COLLECT_SET(CASE WHEN m.support_level = 'Primary' THEN c.capability_name END)) AS primary_capabilities,
        CONCAT_WS(', ', COLLECT_SET(c.capability_name)) AS all_capabilities,
        a.quality_score,
        current_timestamp() AS product_generated_at
    FROM adoit_product.silver.applications a
    LEFT JOIN adoit_product.bronze.raw_app_capability_map m ON a.app_id = m.app_id
    LEFT JOIN adoit_product.silver.business_capabilities c ON m.capability_id = c.capability_id
    GROUP BY a.app_id, a.app_name, a.app_owner, a.department,
             a.lifecycle_status, a.criticality, a.technology_stack,
             a.is_cloud_hosted, a.app_age_years, a.quality_score
) AS source
ON target.app_id = source.app_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [ ]:
%sql
SELECT app_id, app_name, app_owner, department, criticality, lifecycle_status,
       is_cloud_hosted, ROUND(app_age_years, 1) AS age_yrs, capability_count,
       primary_capabilities, quality_score
FROM adoit_product.gold.application_landscape
ORDER BY criticality, app_name

## ✅ Pipeline Complete
- **Bronze**: 10 applications + 10 capabilities + 13 mappings
- **Silver**: 10 applications (enriched) + 10 capabilities
- **Gold**: 10 application landscape records (joined 360° view)

Next: Run `02_Adoit_Governance` for all data product characteristics.